#  1.- Atajos de teclado 


In [ ]:
# Ctrl + s: guardar
# Ctrl + Alt + Enter: Compilar el código
# ... 


# 2.- Instalar DuckDB

In [ ]:
%pip install duckdb


# 3.- Importar DuckDB

In [1]:
import os 
import duckdb


# 4.- Establecer el directorio de trabajo

In [2]:
print(f"La ruta inicial {os.getcwd()}")
os.chdir(r"D:\econometrics\sql\projects\project_004")
print(f"La nueva ruta definida {os.getcwd()}")

La ruta inicial d:\econometrics\sql\projects\project_004\3.queries
La nueva ruta definida D:\econometrics\sql\projects\project_004


# 5.- Leer un CSV directamente 
Aquí aparece una de las grandes ventajas de DuckDB.

In [3]:
# Option 1
duckdb.sql("""
    select 
        *
    from '2.data/actividades.csv'
    limit 5
""").show()


┌──────────────┬─────────────────────┬──────────────────────────────┬────────┐
│ actividad_id │   fecha_envio_utc   │             area             │ semana │
│   varchar    │      timestamp      │           varchar            │ int64  │
├──────────────┼─────────────────────┼──────────────────────────────┼────────┤
│ ACT001       │ 2026-08-17 23:00:00 │ Desarrollo Personal y Social │      1 │
│ ACT002       │ 2026-08-18 23:00:00 │ Comunicación                 │      1 │
│ ACT003       │ 2026-08-19 23:00:00 │ Matemática                   │      1 │
│ ACT004       │ 2026-08-20 23:00:00 │ Desarrollo Personal y Social │      1 │
│ ACT005       │ 2026-08-21 23:00:00 │ Comunicación                 │      1 │
└──────────────┴─────────────────────┴──────────────────────────────┴────────┘



In [4]:
# Option 2
duckdb.sql("""
    select 
        *
    from read_csv('2.data/actividades.csv')
    limit 5
""").show()



┌──────────────┬─────────────────────┬──────────────────────────────┬────────┐
│ actividad_id │   fecha_envio_utc   │             area             │ semana │
│   varchar    │      timestamp      │           varchar            │ int64  │
├──────────────┼─────────────────────┼──────────────────────────────┼────────┤
│ ACT001       │ 2026-08-17 23:00:00 │ Desarrollo Personal y Social │      1 │
│ ACT002       │ 2026-08-18 23:00:00 │ Comunicación                 │      1 │
│ ACT003       │ 2026-08-19 23:00:00 │ Matemática                   │      1 │
│ ACT004       │ 2026-08-20 23:00:00 │ Desarrollo Personal y Social │      1 │
│ ACT005       │ 2026-08-21 23:00:00 │ Comunicación                 │      1 │
└──────────────┴─────────────────────┴──────────────────────────────┴────────┘



# 6.- Explore tables using basic queries

## 6.1.- Query 1: Todos los campos 

In [5]:
duckdb.sql(""" select * from '2.data/instituciones.csv' limit 3""").show()
duckdb.sql(""" select * from '2.data/grupos.csv' limit 3 """).show()
duckdb.sql(""" select * from '2.data/participantes.csv' limit 3""").show()
duckdb.sql(""" select * from '2.data/interacciones.csv' limit 3""").show()
duckdb.sql(""" select * from '2.data/actividades.csv' limit 3""").show()



┌────────────────┬─────────┬─────────┐
│ institucion_id │  ugel   │  zona   │
│    varchar     │ varchar │ varchar │
├────────────────┼─────────┼─────────┤
│ IE-A01         │ UGEL A  │ urbana  │
│ IE-A02         │ UGEL A  │ urbana  │
│ IE-A03         │ UGEL A  │ rural   │
└────────────────┴─────────┴─────────┘

┌──────────┬────────────────┬───────┬────────────┐
│ grupo_id │ institucion_id │ edad  │ docente_id │
│ varchar  │    varchar     │ int64 │  varchar   │
├──────────┼────────────────┼───────┼────────────┤
│ G-A01-4  │ IE-A01         │     4 │ D-A01-4    │
│ G-A01-5  │ IE-A01         │     5 │ D-A01-5    │
│ G-A02-4  │ IE-A02         │     4 │ D-A02-4    │
└──────────┴────────────────┴───────┴────────────┘

┌─────────────────┬──────────┬─────────┬───────────────┬────────────┐
│ participante_id │ grupo_id │   rol   │ telefono_hash │ fecha_alta │
│     varchar     │ varchar  │ varchar │    varchar    │    date    │
├─────────────────┼──────────┼─────────┼───────────────┼────────────

## 6.2.-  Query 2: Filas por tabla (esperado: 24, 48, 1101, 30, 7077)

In [6]:

duckdb.sql(
    """
        select
            'instituciones' as tabla,  
            count(*) as filas 
        from '2.data/instituciones.csv'
        union all select 'grupos',        count(*) from '2.data/grupos.csv'
        union all select 'participantes', count(*) from '2.data/participantes.csv'
        union all select 'actividades',   count(*) from '2.data/actividades.csv'
        union all select 'interacciones', count(*) from '2.data/interacciones.csv';
    """
).show()


┌───────────────┬───────┐
│     tabla     │ filas │
│    varchar    │ int64 │
├───────────────┼───────┤
│ instituciones │    24 │
│ grupos        │    48 │
│ participantes │  1101 │
│ actividades   │    30 │
│ interacciones │  7077 │
└───────────────┴───────┘



# 7.- Create auxiliary tables

## 7.1.- Table 1: Calendario de semanas (lunes a domingo). fin_semana = domingo.

In [7]:
aux_semanas = duckdb.sql(
    """ 
        create or replace table aux_semanas as 
        select
            semana, 
            cast(inicio_semana as date) as inicio_semana,
            cast(fin_semana as date) as fin_semana
        from (values (1,'2026-08-17','2026-08-23'),
                     (2,'2026-08-24','2026-08-30'),
                     (3,'2026-08-31','2026-09-06'),
                     (4,'2026-09-07','2026-09-13'),
                     (5,'2026-09-14','2026-09-20'),
                     (6,'2026-09-21','2026-09-27')) as s(semana, inicio_semana, fin_semana);
    """
)

In [8]:
duckdb.sql("""
    select * from aux_semanas;
""").show()


┌────────┬───────────────┬────────────┐
│ semana │ inicio_semana │ fin_semana │
│ int32  │     date      │    date    │
├────────┼───────────────┼────────────┤
│      1 │ 2026-08-17    │ 2026-08-23 │
│      2 │ 2026-08-24    │ 2026-08-30 │
│      3 │ 2026-08-31    │ 2026-09-06 │
│      4 │ 2026-09-07    │ 2026-09-13 │
│      5 │ 2026-09-14    │ 2026-09-20 │
│      6 │ 2026-09-21    │ 2026-09-27 │
└────────┴───────────────┴────────────┘



## 7.2.- Table 2: Familias únicas (1 fila por PERSONA = telefono_hash). Fecha de alta MÁS ANTIGUA. Sin docentes.

In [9]:

aux_familias = duckdb.sql(
    """
    create or replace table aux_familias as 
    select 
        p.telefono_hash   as persona_id,
        min(p.fecha_alta) as fecha_alta,
        min(p.grupo_id)   as grupo_id,   
        min(i.ugel)       as ugel
    from '2.data/participantes.csv' p
    inner join '2.data/grupos.csv' g        on (g.grupo_id       = p.grupo_id)
    inner join '2.data/instituciones.csv' i on (i.institucion_id = g.institucion_id)
    where p.rol = 'familia'
    group by p.telefono_hash;
    """
)


In [10]:
duckdb.sql(
    """
        select * from aux_familias;
    """
).show()



┌────────────┬────────────┬──────────┬─────────┐
│ persona_id │ fecha_alta │ grupo_id │  ugel   │
│  varchar   │    date    │ varchar  │ varchar │
├────────────┼────────────┼──────────┼─────────┤
│ 9768bb9a63 │ 2026-08-07 │ G-C03-5  │ UGEL C  │
│ ef3bac060b │ 2026-08-06 │ G-C04-5  │ UGEL C  │
│ 63363df707 │ 2026-08-13 │ G-A04-5  │ UGEL A  │
│ 80d96cbe5a │ 2026-08-09 │ G-A02-4  │ UGEL A  │
│ eae079ad77 │ 2026-08-07 │ G-C08-4  │ UGEL C  │
│ c5d3c8e444 │ 2026-08-07 │ G-A06-5  │ UGEL A  │
│ 67d78961c7 │ 2026-08-06 │ G-C04-5  │ UGEL C  │
│ 5adcafbb3f │ 2026-08-06 │ G-C06-4  │ UGEL C  │
│ 16457dec14 │ 2026-08-08 │ G-A02-5  │ UGEL A  │
│ 4be2d7fe1c │ 2026-08-05 │ G-C04-4  │ UGEL C  │
│     ·      │     ·      │    ·     │   ·     │
│     ·      │     ·      │    ·     │   ·     │
│     ·      │     ·      │    ·     │   ·     │
│ 1acbe8992a │ 2026-08-05 │ G-B07-4  │ UGEL B  │
│ cc69b4acdd │ 2026-08-09 │ G-A08-4  │ UGEL A  │
│ 00c8dae332 │ 2026-08-06 │ G-C01-5  │ UGEL C  │
│ 1b2577900b │ 2026-